# 03 - Modèle Baseline (NCF sans contexte)

Ce notebook entraîne un modèle de recommandation de référence sur MovieLens 100K.
Il utilise un NCF simple (`src/models.py`) et traite les interactions comme un problème de feedback binaire.

Objectif : établir un point de référence clair pour mesurer l'impact des features contextuelles dans les notebooks suivants.


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader

sys.path.append(str((Path('..') / 'src').resolve()))
from models import NCF
from metrics import hit_rate, ndcg, mrr
from data_utils import load_movielens_100k, encode_ids

print('Python:', sys.version.split()[0])
print('pandas :', pd.__version__)
print('numpy :', np.__version__)
print('torch :', torch.__version__)


Python: 3.12.13
pandas : 3.0.3
numpy : 2.4.6
torch : 2.12.0+cu130


## Chargement des données

Nous utilisons MovieLens 100K comme dataset de baseline. Si le fichier traité n'existe pas, nous chargeons et encodons directement le dataset brut.


In [2]:
NOTEBOOK_ROOT = Path('.')
RAW_ROOT = (NOTEBOOK_ROOT / '..' / 'data' / 'raw' / 'movielens' / 'ml-100k').resolve()
PROCESSED_PATH = (NOTEBOOK_ROOT / '..' / 'data' / 'processed' / 'movielens_100k' / 'interactions.parquet').resolve()
print('RAW_ROOT =', RAW_ROOT)
print('PROCESSED_PATH =', PROCESSED_PATH)

if PROCESSED_PATH.exists():
    df = pd.read_parquet(PROCESSED_PATH)
    print('Loaded processed file:', PROCESSED_PATH)
else:
    df = load_movielens_100k(RAW_ROOT)
    df = encode_ids(df)
    print('Loaded raw MovieLens 100K and encoded IDs')

print('Dataset shape:', df.shape)
print(df.columns.tolist())
print(df.head(3).to_string(index=False))


RAW_ROOT = /home/mrtds/Documents/my_projects/context-aware-recsys/data/raw/movielens/ml-100k
PROCESSED_PATH = /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/interactions.parquet
Loaded processed file: /home/mrtds/Documents/my_projects/context-aware-recsys/data/processed/movielens_100k/interactions.parquet
Dataset shape: (99287, 19)
['user_id', 'item_id', 'rating', 'timestamp', 'user_id_encoded', 'item_id_encoded', 'hour_of_day', 'day_of_week', 'is_weekend', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'time_since_last_interaction', 'session_id', 'session_length', 'session_position', 'session_position_norm', 'time_since_last_interaction_log']
 user_id  item_id  rating           timestamp  user_id_encoded  item_id_encoded  hour_of_day  day_of_week  is_weekend  hour_sin  hour_cos   dow_sin   dow_cos  time_since_last_interaction  session_id  session_length  session_position  session_position_norm  time_since_last_interaction_log
     259      255     

## Préparation des labels

Nous convertissons les notes en feedback binaire. Cette approche permet d'entraîner un NCF rapide et comparable avec des modèles implicites.


In [3]:
df['label'] = (df['rating'] >= 4).astype(int)
print(df['label'].value_counts(normalize=True))

n_users = df['user_id_encoded'].nunique() if 'user_id_encoded' in df.columns else df['user_id'].nunique()
n_items = df['item_id_encoded'].nunique() if 'item_id_encoded' in df.columns else df['item_id'].nunique()

if 'user_id_encoded' not in df.columns or 'item_id_encoded' not in df.columns:
    df = encode_ids(df)
    n_users = df['user_id_encoded'].nunique()
    n_items = df['item_id_encoded'].nunique()

print('Unique users:', n_users)
print('Unique items:', n_items)
print('Sample interactions:')
print(df[['user_id_encoded', 'item_id_encoded', 'rating', 'label']].head())


label
1    0.555612
0    0.444388
Name: proportion, dtype: float64
Unique users: 943
Unique items: 1349
Sample interactions:
   user_id_encoded  item_id_encoded  rating  label
0              258              253       4      1
1              258              284       4      1
2              258              296       4      1
3              258              183       4      1
4              258              171       4      1


## Split train/test

Nous utilisons un split simple aléatoire. Pour un benchmark plus solide, on pourra remplacer par un split temporel ultérieurement.


In [4]:
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
print('train shape:', train_df.shape)
print('test shape:', test_df.shape)


train shape: (79429, 20)
test shape: (19858, 20)


## Dataset PyTorch

Le dataset renvoie les indices utilisateur/item et le label binaire pour l'entraînement.


In [5]:
class MovielensDataset(Dataset):
    def __init__(self, df):
        self.user_ids = torch.tensor(df['user_id_encoded'].values, dtype=torch.long)
        self.item_ids = torch.tensor(df['item_id_encoded'].values, dtype=torch.long)
        self.labels = torch.tensor(df['label'].values, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.item_ids[idx], self.labels[idx]

train_dataset = MovielensDataset(train_df)
test_dataset = MovielensDataset(test_df)

batch_size = 512
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print('Train batches:', len(train_loader))
print('Test batches:', len(test_loader))


Train batches: 156
Test batches: 39


## Modèle baseline

Nous construisons un NCF simple avec `embed_dim=32` et plusieurs couches MLP.


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = NCF(num_users=n_users, num_items=n_items, embed_dim=32, mlp_layers=[64, 32, 16], dropout=0.2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.BCEWithLogitsLoss()

print(model)


Device: cpu
NCF(
  (embedding_user_gmf): Embedding(943, 32)
  (embedding_item_gmf): Embedding(1349, 32)
  (embedding_user_mlp): Embedding(943, 32)
  (embedding_item_mlp): Embedding(1349, 32)
  (mlp): Sequential(
    (0): Linear(in_features=64, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.2, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=32, out_features=16, bias=True)
    (7): ReLU()
    (8): Dropout(p=0.2, inplace=False)
  )
  (output_layer): Linear(in_features=48, out_features=1, bias=True)
)


## Entraînement

Boucle d'entraînement classique pour minimiser la perte binaire.


In [7]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    for user_batch, item_batch, labels in loader:
        user_batch = user_batch.to(device)
        item_batch = item_batch.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(user_batch, item_batch)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * labels.size(0)

    return total_loss / len(loader.dataset)

epochs = 8
history = []
for epoch in range(1, epochs + 1):
    loss = train_epoch(model, train_loader, optimizer, criterion, device)
    history.append(loss)
    print(f'Epoch {epoch}/{epochs} - train loss: {loss:.4f}')


Epoch 1/8 - train loss: 0.6430
Epoch 2/8 - train loss: 0.5691
Epoch 3/8 - train loss: 0.5486
Epoch 4/8 - train loss: 0.5196
Epoch 5/8 - train loss: 0.4838
Epoch 6/8 - train loss: 0.4466
Epoch 7/8 - train loss: 0.4114
Epoch 8/8 - train loss: 0.3769


## Évaluation ranking

Nous évaluons le modèle sur test avec Hit Rate@10, NDCG@10 et MRR.


In [8]:
all_item_ids = torch.arange(n_items, dtype=torch.long, device=device)

def evaluate_ranking(model, df, k=10):
    model.eval()
    hits, ndcgs, mrrs = [], [], []
    with torch.no_grad():
        for _, row in df.iterrows():
            user_id = torch.tensor([row['user_id_encoded']], dtype=torch.long, device=device)
            item_id = int(row['item_id_encoded'])
            user_batch = user_id.expand(n_items)
            logits = model(user_batch, all_item_ids).cpu().numpy()
            ranking = np.argsort(-logits)
            if item_id in ranking[:k]:
                hits.append(1)
            else:
                hits.append(0)
            ndcgs.append(ndcg(ranking, item_id, k))
            mrrs.append(mrr(ranking, item_id))
    return np.mean(hits), np.mean(ndcgs), np.mean(mrrs)

hit10, ndcg10, mrr_score = evaluate_ranking(model, test_df, k=10)
print(f'Hit Rate@10: {hit10:.4f}')
print(f'NDCG@10: {ndcg10:.4f}')
print(f'MRR: {mrr_score:.4f}')


Hit Rate@10: 0.0248
NDCG@10: 0.0119
MRR: 0.0140


## Sauvegarde du modèle baseline

Nous sauvegardons le modèle entraîné pour comparaison future.


In [9]:
RESULTS_DIR = (NOTEBOOK_ROOT / '..' / 'results' / 'models').resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = RESULTS_DIR / 'baseline_ncf_ml100k.pt'
torch.save(model.state_dict(), MODEL_PATH)
print('Saved baseline model to', MODEL_PATH)


Saved baseline model to /home/mrtds/Documents/my_projects/context-aware-recsys/results/models/baseline_ncf_ml100k.pt


## Résumé

Ce notebook présente une baseline NCF sur MovieLens 100K.
La comparaison de ces métriques avec un modèle contextuel permettra de quantifier le gain apporté par les features temporelles et de session.
